In [ ]:
!pip install faiss-gpu-cu12
import pandas as pd
import numpy as np
import faiss


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.4/48.4 MB 12.6 MB/s eta 0:00:00


In [ ]:
!pip install pyspark -q
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

In [ ]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType, TimestampType

In [ ]:
spark = SparkSession.builder \
    .appName("HM_Visual") \
    .config("spark.driver.memory", "8g") \
    .config("spark.sql.execution.arrow.pyspark.enabled", "true") \
    .getOrCreate()

In [ ]:
!nvidia-smi

Sat Apr  4 05:27:41 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   41C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# =====================================================================
# BƯỚC 1: DÙNG SPARK TÌM MÓN ĐỒ CUỐI CÙNG TRONG 7 TUẦN (DATA LỌC)
# =====================================================================
print("1. Đang trích xuất món đồ cuối cùng của khách hàng...")

# Sử dụng biến 'df' đã được Pandas đọc thành công ở ô trước đó
# Chuyển đổi từ Pandas sang Spark DataFrame
df_trans = spark.createDataFrame(df)

# Tính toán khung thời gian 7 tuần (bỏ 1 tuần cuối cùng làm Test)
from datetime import timedelta

# Tìm ngày lớn nhất
max_date_row = df_trans.select(F.max("t_dat")).collect()[0][0]

# Đảm bảo max_date là kiểu datetime để tính toán
if isinstance(max_date_row, str):
    max_date = pd.to_datetime(max_date_row)
    df_trans = df_trans.withColumn("t_dat_date", F.to_date("t_dat"))
else:
    max_date = max_date_row
    df_trans = df_trans.withColumn("t_dat_date", F.col("t_dat"))

end_train = max_date - pd.Timedelta(days=7)
start_train = end_train - pd.Timedelta(days=49)

# Lọc giao dịch trong 6 tuần và lấy dòng mới nhất của mỗi customer
last_items_spark = df_trans.filter((F.col("t_dat_date") >= start_train.date()) & (F.col("t_dat_date") < end_train.date())) \
    .orderBy("t_dat_date", ascending=False) \
    .dropDuplicates(["customer_id"]) \
    .select("customer_id", "article_id")

# Đưa bảng kết quả thu gọn về Pandas để dễ dàng thao tác với FAISS
last_items_pd = last_items_spark.toPandas()
print(f"   -> Đã tìm được {len(last_items_pd)} khách hàng có giao dịch trong 7 tuần này.")

1. Đang trích xuất món đồ cuối cùng của khách hàng...
   -> Đã tìm được 352409 khách hàng có giao dịch trong 7 tuần này.


In [ ]:
# BƯỚC 2: XÂY DỰNG FAISS INDEX & LẤY ĐIỂM SỐ CLIP
# =====================================================================
print("2. Đang tải vector CLIP và xây dựng bộ máy tìm kiếm FAISS...")
vectors_df = pd.read_parquet("/content/drive/MyDrive/Project Data/output_vectors.parquet")

article_ids = vectors_df['article_id'].values
vectors_matrix = np.stack(vectors_df['clip_vector'].values).astype('float32')

# Chuẩn hóa vector để Inner Product tương đương với Cosine Similarity
faiss.normalize_L2(vectors_matrix)

# Khởi tạo và nạp dữ liệu vào FAISS
index = faiss.IndexFlatIP(vectors_matrix.shape[1])
index.add(vectors_matrix)

# Lọc bỏ những món đồ khách mua mà không có ảnh/vector trong kho CLIP
valid_last_items = last_items_pd[last_items_pd['article_id'].isin(article_ids)].copy()

# Tra cứu siêu tốc vector của các món đồ cuối cùng
vector_dict = {aid: i for i, aid in enumerate(article_ids)}
query_indices = [vector_dict[aid] for aid in valid_last_items['article_id']]
query_vectors = vectors_matrix[query_indices]

print("   -> Đang truy vấn tìm 50 sản phẩm tương đồng & tính điểm (Cosine)...")
# Lấy ra 51 kết quả để loại bỏ Top 1 (là chính nó)
# Lấy ĐỒNG THỜI điểm số (scores) và vị trí (indices)
scores, indices = index.search(query_vectors, 51)

2. Đang tải vector CLIP và xây dựng bộ máy tìm kiếm FAISS...
   -> Đang truy vấn tìm 50 sản phẩm tương đồng & tính điểm (Cosine)...


In [ ]:
# BƯỚC 3: PHẲNG HÓA DỮ LIỆU (LONG FORMAT) CHO XGBOOST
# =====================================================================
print("3. Đang chuyển đổi dữ liệu sang định dạng Long Format...")

cust_ids = valid_last_items['customer_id'].values
all_rows = []

# Vòng lặp chuyển từ mảng (Array) sang từng dòng rời rạc (Rows)
for i in range(len(cust_ids)):
    customer = cust_ids[i]

    # Bỏ qua vị trí số 0 (chính món đồ đó), lấy 50 món còn lại
    sim_indices = indices[i][1:]
    sim_scores = scores[i][1:]

    # Tạo các bộ (Tuple): Customer - Article - Score
    for idx, score in zip(sim_indices, sim_scores):
        all_rows.append((customer, article_ids[idx], float(score)))

# Tạo DataFrame phẳng
visual_candidates_long = pd.DataFrame(all_rows, columns=['customer_id', 'article_id', 'clip_score'])


# =====================================================================

3. Đang chuyển đổi dữ liệu sang định dạng Long Format...


In [ ]:
# BƯỚC 4: LƯU BÁU VẬT LÊN DRIVE
# =====================================================================
save_path = "/content/drive/MyDrive/Project Data/visual_candidates_7weeks_long.parquet"
visual_candidates_long.to_parquet(save_path, index=False)

print(f" HOÀN TẤT XUẤT SẮC!")
print(f"   -> Đã tạo ra {len(visual_candidates_long)} dòng dữ liệu.")
print(f"   -> Cấu trúc bảng: customer_id | article_id | clip_score")
print(f"   -> Đã lưu an toàn tại: {save_path}")

🎉 HOÀN TẤT XUẤT SẮC!
   -> Đã tạo ra 17566050 dòng dữ liệu.
   -> Cấu trúc bảng: customer_id | article_id | clip_score
   -> Đã lưu an toàn tại: /content/drive/MyDrive/Project Data/visual_candidates_7weeks_long.parquet


In [ ]:
import pandas as pd

# Đọc file từ Drive
df_check = pd.read_parquet("/content/drive/MyDrive/Project Data/visual_candidates_7weeks_long.parquet")

# Hiển thị 60 dòng đầu
display(df_check.head(60))

# Kiểm tra tổng số dòng và cột
print(f"Tổng số khách hàng trong file: {len(df_check)}")

,customer_id,article_id,clip_score
0,00006413d8573cd20ed7128e53b7b13819fe5cfc2d801f...,0932558001,0.968458
1,00006413d8573cd20ed7128e53b7b13819fe5cfc2d801f...,0632223006,0.965181
2,00006413d8573cd20ed7128e53b7b13819fe5cfc2d801f...,0908467001,0.964455
3,00006413d8573cd20ed7128e53b7b13819fe5cfc2d801f...,0801252001,0.962836
4,00006413d8573cd20ed7128e53b7b13819fe5cfc2d801f...,0905121001,0.961691
5,00006413d8573cd20ed7128e53b7b13819fe5cfc2d801f...,0791120002,0.961447
6,00006413d8573cd20ed7128e53b7b13819fe5cfc2d801f...,0905254008,0.961237
7,00006413d8573cd20ed7128e53b7b13819fe5cfc2d801f...,0747349002,0.960343
8,00006413d8573cd20ed7128e53b7b13819fe5cfc2d801f...,0620928012,0.960118
9,00006413d8573cd20ed7128e53b7b13819fe5cfc2d801f...,0889573005,0.960105


Tổng số khách hàng trong file: 17566050


In [ ]:
print("📦 2. Đang đọc dữ liệu (Tự động nhận diện Schema chuẩn)...")
preds_df = spark.read.parquet("/content/drive/MyDrive/Project Data/visual_candidates_7weeks_long.parquet")

# BÍ KÍP: Thêm "/*" vào cuối để ép Spark chui vào ruột thư mục và tự nhận diện Schema chính xác
path_trans = "/content/drive/MyDrive/Project Data/cleaned_transactions.parquet/*"
df_trans = spark.read.parquet(path_trans)

📦 2. Đang đọc dữ liệu (Tự động nhận diện Schema chuẩn)...


ConnectionRefusedError: [Errno 111] Connection refused

In [ ]:
print("⏳ 3. Đang tìm ngày tháng và lọc Tuần 7...")
# Lấy trực tiếp ngày lớn nhất từ cột t_dat_date do bạn kia tạo
max_date_raw = df_trans.select(F.max("t_dat_date")).collect()[0][0]

if max_date_raw is None:
    raise ValueError("LỖI: Spark vẫn không đọc được dữ liệu. Hãy kiểm tra lại file của người bạn!")

# BẢO HIỂM KHOẢNG CÁCH: Ép kiểu để chắc chắn nó trừ được cho timedelta
if isinstance(max_date_raw, str):
    max_date = datetime.datetime.strptime(max_date_raw[:10], "%Y-%m-%d").date()
elif isinstance(max_date_raw, datetime.datetime):
    max_date = max_date_raw.date()
else:
    max_date = max_date_raw # Trường hợp nó đã là kiểu Date chuẩn

print(f"   -> Đã chốt ngày giao dịch cuối cùng là: {max_date}")

# Tính toán Tuần 7
end_train = max_date - datetime.timedelta(days=14)
start_week7 = end_train
end_week7 = end_train + datetime.timedelta(days=7)

start_str = start_week7.strftime("%Y-%m-%d")
end_str = end_week7.strftime("%Y-%m-%d")
print(f"   -> Vùng kiểm tra Tuần 7: {start_str} đến {end_str}")

# Lọc Tuần 7 thực tế
actual_week7 = df_trans.filter((F.col("t_dat_date") >= start_str) & (F.col("t_dat_date") < end_str)) \
    .select("customer_id", "article_id") \
    .dropDuplicates()


⏳ 3. Đang tìm ngày tháng và lọc Tuần 7...
   -> Đã chốt ngày giao dịch cuối cùng là: 2020-09-22
   -> Vùng kiểm tra Tuần 7: 2020-09-08 đến 2020-09-15


In [ ]:
# 4. CHẤM ĐIỂM VỚI BROADCAST JOIN (TUYỆT CHIÊU CHỐNG SẬP RAM)
print("🎯 4. Đang chấm điểm Recall (Sẽ mất khoảng 1 phút)...")
actual_counts = actual_week7.groupBy("customer_id").count().withColumnRenamed("count", "total_actual")

# Dùng F.broadcast(actual_week7) để ép Spark xử lý nhẹ nhàng nhất
hits_df = preds_df.join(F.broadcast(actual_week7), ["customer_id", "article_id"], "inner")
hit_counts = hits_df.groupBy("customer_id").count().withColumnRenamed("count", "hits")

eval_df = actual_counts.join(hit_counts, "customer_id", "left").fillna(0, subset=["hits"])
eval_df = eval_df.withColumn("recall", F.col("hits") / F.col("total_actual"))

# 5. TỔNG HỢP KẾT QUẢ
mean_recall_row = eval_df.select(F.mean("recall").alias("mean_recall")).collect()[0]
mean_recall = mean_recall_row["mean_recall"] if mean_recall_row["mean_recall"] else 0.0

total_users = eval_df.count()
users_with_hits = eval_df.filter(F.col("hits") > 0).count()
hit_rate = users_with_hits / total_users if total_users > 0 else 0.0

print("\n=========================================================")
print(f"🏆 KẾT QUẢ ĐÁNH GIÁ VISUAL RECALL@50 (TUẦN 7)")
print("=========================================================")
print(f"👤 Số lượng khách hàng mua ở Tuần 7: {total_users:,} người")
print(f"📈 Tỉ lệ Hit Rate (Trúng ít nhất 1 món): {hit_rate * 100:.2f}%")
print(f"🔥 ĐIỂM RECALL TRUNG BÌNH:               {mean_recall * 100:.2f}%")
print("=========================================================")

🎯 4. Đang chấm điểm Recall (Sẽ mất khoảng 1 phút)...

🏆 KẾT QUẢ ĐÁNH GIÁ VISUAL RECALL@50 (TUẦN 7)
👤 Số lượng khách hàng mua ở Tuần 7: 74,575 người
📈 Tỉ lệ Hit Rate (Trúng ít nhất 1 món): 1.62%
🔥 ĐIỂM RECALL TRUNG BÌNH:               0.68%
